#  Resource Allocation Model — Sri Lanka District Poverty
### Generates risk percentages per district and distributes budget equitably

**Pipeline Overview**
1. Install & Import dependencies
2. Load & preprocess `Povertylines.csv`
3. Feature Engineering & Normalisation
4. Enhanced Risk Score with Custom Rules
5. Budget Allocation Engine
6. Allocation Report & Visualisation
7. Interactive Budget Input

> Upload `Povertylines.csv` via the Colab Files panel before running.

##  Section 1 — Install & Import Dependencies

In [ ]:
!pip install sentence-transformers scikit-learn matplotlib seaborn pandas numpy --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from sentence_transformers import SentenceTransformer
import pickle
warnings.filterwarnings("ignore")
from sklearn.preprocessing import MinMaxScaler
from matplotlib.patches import Patch

sns.set_theme(style="whitegrid", palette="muted")
print(" Libraries loaded.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")

##  Section 2 — Load & Preprocess Data

Loads `Povertylines.csv`, standardises column names, strips comma-formatted numbers,
and fixes known anomalies (Kilinochchi Gini = 0, Gampaha per-capita income = 269).

In [ ]:
FILE_PATH = "/content/drive/MyDrive/DSGP/poverty/Povertylines.xlsx"
df_raw = pd.read_excel(FILE_PATH)

df = df_raw.copy()
df.columns = (df.columns.str.strip().str.lower()
               .str.replace(r"\s+", "_", regex=True)
               .str.replace(r"[().]", "", regex=True)
               .str.replace(r"_+", "_", regex=True))

rename_map = {
    "mean_household_income_per_month":                 "mean_hh_income",
    "median_household_income_per_month_rs":            "median_hh_income",
    "average_household_size":                          "avg_hh_size",
    "gini_coefficient_income":                         "gini_income", # Corrected key
    "mean_per_capita_income_per_month_rs":             "mean_per_capita_income",
    "mean_household_expenditure_per_month_rs":         "mean_hh_expenditure",
    "median_household_expenditure_per_month_rs":       "median_hh_expenditure",
    "gini_coefficient_expenditure":                    "gini_expenditure", # Corrected key
    "mean_household_per_capita_expenditure_per_month": "mean_hh_per_capita_expenditure",
}
df.rename(columns=rename_map, inplace=True)

for col in [c for c in df.columns if c != "district"]:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce")

# Fix anomalies
median_gini = df.loc[df["gini_income"] > 0, "gini_income"].median()
df.loc[df["gini_income"] == 0, "gini_income"] = median_gini
median_gini_exp = df.loc[df["gini_expenditure"] > 0, "gini_expenditure"].median()
df.loc[df["gini_expenditure"] == 0, "gini_expenditure"] = median_gini_exp
mask = df["district"] == "Gampaha"
df.loc[mask, "mean_per_capita_income"] = (
    df.loc[mask, "mean_hh_income"] / df.loc[mask, "avg_hh_size"]).round(0)

price_cols = [c for c in df.columns if c.startswith("202")]
hh_cols = ["mean_hh_income","median_hh_income","avg_hh_size","gini_income",
           "mean_per_capita_income","mean_hh_expenditure","median_hh_expenditure",
           "gini_expenditure","mean_hh_per_capita_expenditure"]

print(f" Loaded & cleaned: {df.shape[0]} districts × {df.shape[1]} columns")
df[["district"] + hh_cols]


## Section 3 — Feature Engineering & Normalisation

Derives 7 poverty indicators and scales them to `[0, 1]` using Min-Max normalisation.

| Feature | Logic |
|---|---|
| `poverty_proxy` | `1 / mean_per_capita_income` |
| `inequality_score` | `gini_income` |
| `expenditure_burden` | `mean_hh_expenditure / mean_hh_income` |
| `hh_size_pressure` | `avg_hh_size` |
| `price_pressure` | Latest month price index |
| `price_trend` | Linear slope over all monthly values |
| `price_volatility` | Std dev of monthly index |

In [ ]:
scaler = MinMaxScaler()

df["price_volatility"]   = df[price_cols].std(axis=1)
df["poverty_proxy"]      = 1 / df["mean_per_capita_income"]
df["inequality_score"]   = df["gini_income"]
df["expenditure_burden"] = df["mean_hh_expenditure"] / df["mean_hh_income"]
df["hh_size_pressure"]   = df["avg_hh_size"]
df["price_pressure"]     = df[price_cols[-1]]

month_x = np.arange(len(price_cols))
df["price_trend"] = df[price_cols].apply(
    lambda row: np.polyfit(month_x, row.values.astype(float), 1)[0], axis=1)

feature_cols = ["poverty_proxy","inequality_score","expenditure_burden",
                "hh_size_pressure","price_pressure","price_trend","price_volatility"]

df_norm = df.copy()
df_norm[feature_cols] = scaler.fit_transform(df[feature_cols])

DEFAULT_RULES = {
    "poverty_proxy":0.30, "inequality_score":0.20, "expenditure_burden":0.20,
    "price_pressure":0.15, "price_trend":0.10, "hh_size_pressure":0.05,
}

def compute_risk_index(row, rules):
    return round(sum(row[f] * w for f, w in rules.items()), 4)

df_norm["base_risk_index"] = df_norm.apply(lambda r: compute_risk_index(r, DEFAULT_RULES), axis=1)

print(" Features engineered and normalised.")
df_norm[["district"] + feature_cols + ["base_risk_index"]].round(3)

## Section 4 — Enhanced Risk Score with Custom Rules

The enhanced risk score adds **penalty bonuses** on top of the base weighted score
to flag districts with compounding poverty indicators:

| Rule | Condition | Bonus |
|---|---|---|
| **Extreme Poverty** | Per capita income < Rs. 15,000 | +0.10 |
| **High Inequality** | Gini > 0.45 | +0.08 |
| **Expenditure Exceeds Income** | Expenditure/Income > 90% | +0.07 |
| **Large HH + Low Income** | HH size ≥ 4.0 AND income < Rs. 55,000 | +0.06 |
| **Rising Prices in Poor District** | Price trend rising + poverty score > 0.5 | +0.05 |

**Risk Tiers:**
- CRITICAL → score ≥ 0.70
- HIGH → score ≥ 0.55
- MODERATE → score ≥ 0.40
- LOW → score < 0.40

In [ ]:
ALLOCATION_RULES = {
    "base_weights": {
        "poverty_proxy":0.30, "inequality_score":0.20, "expenditure_burden":0.20,
        "price_pressure":0.15, "price_trend":0.10, "hh_size_pressure":0.05,
    },
    "penalties": {
        "extreme_poverty_threshold":    15000,
        "extreme_poverty_bonus":        0.10,
        "high_gini_threshold":          0.45,
        "high_gini_bonus":              0.08,
        "expenditure_burden_threshold": 0.90,
        "expenditure_burden_bonus":     0.07,
        "large_hh_size_threshold":      4.0,
        "large_hh_income_threshold":    55000,
        "large_hh_bonus":               0.06,
        "rising_price_poverty_bonus":   0.05,
    },
    "equity_floor_pct": 0.015,   # 1.5% minimum allocation per district
}

def compute_enhanced_risk_score(row_norm, row_raw, rules):
    """Base weighted score + custom penalty bonuses."""
    p = rules["penalties"]

    # Layer 1: Base score
    base = sum(row_norm[f] * w for f, w in rules["base_weights"].items())

    # Layer 2: Penalty bonuses
    bonus = 0.0
    if row_raw["mean_per_capita_income"] < p["extreme_poverty_threshold"]:
        bonus += p["extreme_poverty_bonus"]
    if row_raw["gini_income"] > p["high_gini_threshold"]:
        bonus += p["high_gini_bonus"]
    if (row_raw["mean_hh_expenditure"] / row_raw["mean_hh_income"]) > p["expenditure_burden_threshold"]:
        bonus += p["expenditure_burden_bonus"]
    if (row_raw["avg_hh_size"] >= p["large_hh_size_threshold"] and
            row_raw["mean_hh_income"] < p["large_hh_income_threshold"]):
        bonus += p["large_hh_bonus"]
    if row_norm["price_trend"] > 0.5 and row_norm["poverty_proxy"] > 0.5:
        bonus += p["rising_price_poverty_bonus"]

    return round(min(base + bonus, 1.0), 4)

def classify_tier(score):
    if score >= 0.70: return "CRITICAL"
    if score >= 0.55: return "HIGH"
    if score >= 0.40: return "MODERATE"
    return "LOW"

df_alloc = df_norm.copy()
df_alloc["enhanced_risk_score"] = [
    compute_enhanced_risk_score(df_norm.loc[i], df.loc[i], ALLOCATION_RULES)
    for i in df_norm.index
]
df_alloc["risk_tier"] = df_alloc["enhanced_risk_score"].apply(classify_tier)

total_risk = df_alloc["enhanced_risk_score"].sum()
df_alloc["risk_pct"] = (df_alloc["enhanced_risk_score"] / total_risk * 100).round(3)

## Section 5 — Budget Allocation Engine

**How allocation works:**

1. **Equity Floor** — Every district receives a guaranteed minimum of **1.5%** of total budget
2. **Proportional Share** — Remaining budget distributed by each district's risk score share
3. **Final Allocation** = Floor Amount + Proportional Share

This ensures even low-risk districts receive some funding while
high-risk districts receive significantly more.

In [ ]:
def allocate_budget(total_budget, df_risk, rules):
    """
    Allocate total_budget across districts based on enhanced risk scores.

    Parameters
    ----------
    total_budget : float   Total budget in Rs.
    df_risk      : df      Must contain 'district', 'enhanced_risk_score', 'risk_pct', 'risk_tier'
    rules        : dict    ALLOCATION_RULES

    Returns
    -------
    pd.DataFrame  Full allocation breakdown sorted by allocation (descending)
    """
    n            = len(df_risk)
    floor_pct    = rules["equity_floor_pct"]
    floor_amount = round(total_budget * floor_pct, 0)
    total_floor  = floor_amount * n
    remaining    = total_budget - total_floor

    out = df_risk[["district","enhanced_risk_score","risk_pct","risk_tier"]].copy()

    risk_sum = out["enhanced_risk_score"].sum()
    out["prop_share"]       = out["enhanced_risk_score"] / risk_sum
    out["prop_allocation"]  = (out["prop_share"] * remaining).round(0)
    out["floor_allocation"] = floor_amount
    out["total_allocation"] = out["floor_allocation"] + out["prop_allocation"]
    out["allocation_pct"]   = (out["total_allocation"] / total_budget * 100).round(3)

    # Per-household allocation
    out["alloc_per_hh"] = (
        out["total_allocation"].values / df[["avg_hh_size"]].values.flatten()
    ).round(0)

    # Fix rounding gap
    diff = total_budget - out["total_allocation"].sum()
    if abs(diff) > 0:
        top_idx = out["enhanced_risk_score"].idxmax()
        out.loc[top_idx, "total_allocation"] += diff

    return out.sort_values("total_allocation", ascending=False).reset_index(drop=True)

print("allocate_budget() function defined.")